In [0]:
dbutils.library.restartPython()

In [0]:
# ===============================
# Setup Python Path for Src Folder
# ===============================
import sys
import os

# Calculate absolute path to Src folder relative to this notebook
src_path = os.path.abspath(os.path.join(os.getcwd(), "../Src"))
if src_path not in sys.path:
    sys.path.append(src_path)

print("Src folder added to Python path:", src_path)

# Import functions from Src
try:
    from DataContractValidator import load_contract, validate_table_contract
    from SilverLayer import create_silver_tables
except ModuleNotFoundError as e:
    print("Error importing modules from Src folder:", e)
    raise

# Load Contract YAML
contract_path = os.path.join(src_path, "Contracts.yaml")
contract = load_contract(contract_path)
print("Contract loaded successfully")

In [0]:
# =============================== 
# Imports & Setup 
# ===============================
from pyspark.sql.functions import *
from pyspark.sql.functions import col
from collections import Counter

# Create schemas if they don't exist
spark.sql("DROP SCHEMA IF EXISTS bronze CASCADE")
spark.sql("DROP SCHEMA IF EXISTS silver CASCADE")
spark.sql("DROP SCHEMA IF EXISTS gold CASCADE")
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")
print("Schemas created (bronze, silver, gold)")

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, when, monotonically_increasing_id, row_number, desc
from pyspark.sql import DataFrame

# ------------------------------
# Helper Functions
# ------------------------------
def normalize_columns(df: DataFrame) -> DataFrame:
    """Lowercase all column names"""
    return df.toDF(*[c.lower() for c in df.columns])

def fix_drug_schema_bronze(df: DataFrame) -> DataFrame:
    """
    Cast all numeric columns to string to avoid write errors in Bronze.
    """
    for c, t in df.dtypes:
        if t in ["int", "bigint", "double", "float", "decimal"]:
            df = df.withColumn(c, col(c).cast("string"))
    
    # Ensure known string columns are strings
    string_columns = ["drug_expiration_date", "nda_number"]
    for c in string_columns:
        if c in df.columns:
            df = df.withColumn(c, col(c).cast("string"))
    
    return df

def fix_null_primary_id(df: DataFrame) -> DataFrame:
    """Fill null primary_id with unique ID"""
    if "primary_id" in df.columns:
        df = df.withColumn(
            "primary_id",
            when(col("primary_id").isNull(), monotonically_increasing_id())
            .otherwise(col("primary_id"))
        )
    return df

def union_all(df_dict: dict) -> DataFrame:
    """Union all DataFrames in a dictionary by year"""
    result = None
    for year, df in sorted(df_dict.items()):
        result = df if result is None else result.unionByName(df, allowMissingColumns=True)
    return result

In [0]:
# ------------------------------
# Configuration
# ------------------------------
years = [2017, 2018, 2019, 2020, 2021, 2022, 2023]  # pick years dynamically
table_types = ["demographics", "drug", "drug_reaction", "patient_outcome"]

tables = {t: {} for t in table_types}

base_path_template = (
    "john_snow_labs_fda_drug_adverse_events_reporting_system_faers_{year}."
    "fda_drug_adverse_events_reporting_system_faers_{year}."
    "fda_adverse_events_reporting_system_{table}_{year}"
)

# ------------------------------
# Load Tables Dynamically
# ------------------------------
for year in years:
    for table in table_types:
        df = spark.table(base_path_template.format(year=year, table=table))
        df = normalize_columns(df)
        tables[table][year] = df

# ------------------------------
# Fix Drug Schema and Null IDs
# ------------------------------
for year in years:
    tables["drug"][year] = fix_drug_schema_bronze(tables["drug"][year])
    if year == 2022:  # keep special fix for 2022 if needed
        tables["drug"][year] = fix_null_primary_id(tables["drug"][year])

# ------------------------------
# Combine All Years
# ------------------------------
Demographics = union_all(tables["demographics"])
Drug = union_all(tables["drug"])
Reaction = union_all(tables["drug_reaction"])
Outcome = union_all(tables["patient_outcome"])

print("All tables loaded, normalized, and combined!")

# ------------------------------
# Keep Latest Demographics Record
# ------------------------------
if "primary_id" in Demographics.columns and "case_version_number" in Demographics.columns:
    windowSpec = Window.partitionBy("primary_id").orderBy(desc("case_version_number"))
    Demographics = (
        Demographics
        .withColumn("row_num", row_number().over(windowSpec))
        .filter(col("row_num") == 1)
        .drop("row_num")
    )

# ------------------------------
# Data Validation
# ------------------------------
print("Drug row counts by year:")
if "year" in Drug.columns:
    Drug.groupBy("year").count().orderBy("year").show()

print("Demographics row counts by year:")
if "year" in Demographics.columns:
    Demographics.groupBy("year").count().orderBy("year").show()

# ------------------------------
# Save Bronze Tables
# ------------------------------
Demographics.write.format("delta").mode("overwrite").saveAsTable("bronze.demographics")
Drug.write.format("delta").mode("overwrite").saveAsTable("bronze.drug")
Reaction.write.format("delta").mode("overwrite").saveAsTable("bronze.reaction")
Outcome.write.format("delta").mode("overwrite").saveAsTable("bronze.outcome")

print("Bronze tables created successfully")

In [0]:
# =============================== 
# Run Data Contracts 
# ===============================
# validate_table_contract(Demographics, "demographics", contract)
# validate_table_contract(Drug, "drug", contract)
# validate_table_contract(Reaction, "reaction", contract)
# validate_table_contract(Outcome, "outcome", contract)
# print("All data contracts passed")

In [0]:
import pyspark.sql.functions as F
# ===============================
# Pipeline Orchestrator
# ===============================

# Load Bronze tables
demographics = spark.table("bronze.demographics")
drug = spark.table("bronze.drug")
reaction = spark.table("bronze.reaction")
outcome = spark.table("bronze.outcome")

# Run Silver transformation
silver_tables = create_silver_tables(demographics, drug, reaction, outcome)

# Save Silver tables
silver_tables["demographics"].write.format("delta").mode("overwrite").saveAsTable("silver.demographics")
silver_tables["drugs"].write.format("delta").mode("overwrite").saveAsTable("silver.drugs")
silver_tables["reactions"].write.format("delta").mode("overwrite").saveAsTable("silver.reactions")
silver_tables["outcomes"].write.format("delta").mode("overwrite").saveAsTable("silver.outcomes")

print("Silver layer pipeline completed")

In [0]:
# ===============================
# Clean Gold Layer – Fixed Ambiguous year_id
# ===============================

from pyspark.sql.functions import countDistinct, col, when, row_number
from pyspark.sql.window import Window

# ------------------------------
# Load Silver tables once
# ------------------------------
silver_demographics = spark.table("silver.demographics")
silver_drugs = spark.table("silver.drugs")
silver_reactions = spark.table("silver.reactions")
silver_outcomes = spark.table("silver.outcomes")

# ------------------------------
# Cast join keys consistently
# ------------------------------
for df in [silver_drugs, silver_reactions, silver_demographics, silver_outcomes]:
    df = df.withColumn("primary_id", col("primary_id").cast("string")) \
           .withColumn("case_id", col("case_id").cast("string")) \
           .withColumn("year", col("year").cast("int"))

# ------------------------------
# Map year once to year_id
# ------------------------------
year_mapping = {2020: 1, 2022: 2, 2023: 3}

def map_year(df):
    return df.withColumn(
        "year_id",
        when(col("year") == 2020, 1)
        .when(col("year") == 2022, 2)
        .when(col("year") == 2023, 3)
    )

silver_drugs = map_year(silver_drugs)
silver_reactions = map_year(silver_reactions)
silver_demographics = map_year(silver_demographics)
silver_outcomes = map_year(silver_outcomes)

In [0]:
# ------------------------------
# 1️⃣ Top 5 Drugs with Highest Adverse Events (All Years)
# ------------------------------
gold_top5_drugs_overall = (
    silver_drugs
    .join(silver_reactions.drop("year_id"), ["primary_id", "case_id"], "left")  # drop duplicate year_id
    .groupBy("drug_name")
    .agg(countDistinct("primary_id").alias("num_reports"))
    .orderBy(col("num_reports").desc())
    .limit(5)
)

gold_top5_drugs_overall.write.format("delta").mode("overwrite") \
    .saveAsTable("gold.top5_drugs_overall")

In [0]:
# ------------------------------
# 2️⃣ Top 5 Drugs Over Time (Yearly)
# ------------------------------
drug_event_counts_yearly = (
    silver_drugs
    .join(silver_reactions.drop("year_id"), ["primary_id", "case_id", "year"], "left")
    .filter(col("year_id").isNotNull())
    .groupBy("year", "drug_name")
    .agg(countDistinct("primary_id").alias("num_reports"))
)

total_reports_per_drug = (
    drug_event_counts_yearly
    .groupBy("drug_name")
    .agg({"num_reports": "sum"})
    .withColumnRenamed("sum(num_reports)", "total_reports")
)

top5_drugs = (
    total_reports_per_drug
    .orderBy(col("total_reports").desc())
    .limit(5)
    .select("drug_name")
)

gold_top5_drug_counts_yearly = (
    drug_event_counts_yearly
    .join(top5_drugs, on="drug_name", how="inner")
    .orderBy("drug_name", "year")
)

gold_top5_drug_counts_yearly.write.format("delta").mode("overwrite") \
    .saveAsTable("gold.drug_event_counts_top5")

In [0]:
# ------------------------------
# 3️⃣ Most Common Drug Reactions (Top 10 per Drug per Year)
# ------------------------------
drug_reactions = (
    silver_drugs
    .join(silver_reactions.drop("year_id"), ["primary_id", "case_id", "year"], "left")
    .join(top5_drugs, "drug_name", "inner")
    .filter(col("year").isin(2020, 2022, 2023))
)

reaction_counts = (
    drug_reactions
    .groupBy("year", "drug_name", "preferred_term_for_event")
    .agg(countDistinct("primary_id").alias("num_reports"))
)

window_spec = Window.partitionBy("year", "drug_name").orderBy(col("num_reports").desc())

gold_top5_drug_reaction_trends = (
    reaction_counts
    .withColumn("rank", row_number().over(window_spec))
    .filter(col("rank") <= 10)
)

gold_top5_drug_reaction_trends.write.format("delta").mode("overwrite") \
    .saveAsTable("gold.top5_drug_reaction_trends")

In [0]:
# ------------------------------
# 4️⃣ Severe Outcomes by Drug
# ------------------------------
severe_codes = ["DE", "HO", "LT"]

drug_outcomes = (
    silver_drugs
    .join(silver_outcomes.drop("year_id"), ["primary_id", "case_id", "year"], "left")
    .join(top5_drugs, "drug_name", "inner")
    .filter(col("year_id").isNotNull() & col("patient_outcome").isin(severe_codes))
)

gold_top5_drug_severe_outcomes = (
    drug_outcomes
    .groupBy("year_id", "drug_name", "patient_outcome")
    .agg(countDistinct("primary_id").alias("num_reports"))
    .orderBy("drug_name", "year_id", "patient_outcome")
)

gold_top5_drug_severe_outcomes.write.format("delta").mode("overwrite") \
    .saveAsTable("gold.top5_drug_severe_outcomes")

In [0]:
# ------------------------------
# 5️⃣ Adverse Events by Age Group
# ------------------------------
drug_demographics = (
    silver_drugs
    .join(silver_demographics.drop("year_id"), ["primary_id", "case_id", "year"], "left")
    .filter(col("year_id").isNotNull())
    .join(top5_drugs, "drug_name", "inner")
)

gold_top5_drug_age_groups = (
    drug_demographics
    .groupBy("drug_name", "age_group_readable")
    .agg(countDistinct("primary_id").alias("num_reports"))
    .orderBy("drug_name", "num_reports")
)

gold_top5_drug_age_groups.write.format("delta").mode("overwrite") \
    .saveAsTable("gold.top5_drug_age_groups")

In [0]:
# ------------------------------
# 6️⃣ Adverse Events by Gender
# ------------------------------
drug_gender = (
    silver_drugs
    .join(silver_demographics.drop("year_id"), ["primary_id", "case_id", "year"], "left")
    .filter(col("year_id").isNotNull())
    .join(top5_drugs, "drug_name", "inner")
)

gold_top5_drug_gender = (
    drug_gender
    .filter(col("gender_of_patient").isin("M", "F"))
    .groupBy("drug_name", "gender_of_patient")
    .agg(countDistinct("primary_id").alias("num_reports"))
    .orderBy("drug_name", "gender_of_patient")
)

gold_top5_drug_gender.write.format("delta").mode("overwrite") \
    .saveAsTable("gold.top5_drug_gender")

In [0]:
# ------------------------------
# Display all Gold tables at the end
# ------------------------------
display(gold_top5_drugs_overall)
display(gold_top5_drug_counts_yearly)
display(gold_top5_drug_reaction_trends)
display(gold_top5_drug_severe_outcomes)
display(gold_top5_drug_age_groups)
display(gold_top5_drug_gender)